# POC: Fine-tune Llama-Guard-3-8B → Merge → GGUF Q4_K_M

Flow:

```
Base model BF16
      +
LoRA adapter
      ↓
Merge adapter into base model
      ↓
Save merged model as BF16/FP16
      ↓
Convert to GGUF
      ↓
Quantize to Q4_K_M
```

**Colab tip:** use a GPU with enough VRAM (A100 / L4 recommended). `meta-llama/Llama-Guard-3-8B` is gated — accept the license on Hugging Face and set `HF_TOKEN`.

## Step 1: Install libraries

In [ ]:
!pip install -q transformers datasets peft trl accelerate bitsandbytes huggingface_hub sentencepiece protobuf

## Step 1.1: Hugging Face login (gated model)

In [ ]:
from huggingface_hub import login

# Colab: Runtime → Secrets → add HF_TOKEN
# Or paste token below
try:
    from google.colab import userdata
    login(token=userdata.get("HF_TOKEN"))
except Exception:
    login()  # interactive prompt

## Step 2: Load base model (BF16)

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "meta-llama/Llama-Guard-3-8B"
ADAPTER_DIR = "lora-adapter"
MERGED_DIR = "model-merged"
GGUF_F16 = "model-f16.gguf"
GGUF_Q4 = "model-q4_k_m.gguf"

assert torch.cuda.is_available(), "GPU required for BF16 8B training"
print(f"GPU: {torch.cuda.get_device_name(0)}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.bfloat16,
    device_map="auto",
)
model.config.use_cache = False

print(f"Loaded {MODEL_ID} in BF16")

## Step 3: Dataset preparing

In [ ]:
from datasets import Dataset

# Tiny POC dataset — replace with your real Llama-Guard style labels
raw_data = [
    {"instruction": "Say Hello", "output": "Hello!"},
    {"instruction": "2+2", "output": "4"},
]


def format_example(example):
    messages = [
        {"role": "user", "content": example["instruction"]},
        {"role": "assistant", "content": example["output"]},
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}


dataset = Dataset.from_list(raw_data).map(format_example)
print(dataset)
print("\nExample formatted prompt:\n", dataset[0]["text"])

## Step 4: Fine-tune (Base BF16 + LoRA)

In [ ]:
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    bias="none",
    task_type="CAUSAL_LM",
)

training_args = SFTConfig(
    output_dir="./output",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=3,
    learning_rate=2e-4,
    logging_steps=1,
    save_strategy="no",
    bf16=True,
    gradient_checkpointing=True,
    max_length=512,
    dataset_text_field="text",
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    peft_config=lora_config,
    processing_class=tokenizer,
)

trainer.train()
trainer.save_model(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f"Saved LoRA adapter → {ADAPTER_DIR}/")

## Step 5: Merge LoRA adapter into base model → save BF16

In [ ]:
import gc
from peft import PeftModel

# Free trainer / peft-wrapped model before reloading clean base
del trainer
del model
gc.collect()
torch.cuda.empty_cache()

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.bfloat16,
    device_map="auto",
)

model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
merged_model = model.merge_and_unload()

merged_model.save_pretrained(MERGED_DIR, safe_serialization=True)
tokenizer.save_pretrained(MERGED_DIR)

print(f"Merged BF16 model saved → {MERGED_DIR}/")

## Step 6: Convert merged model → GGUF (F16)

In [ ]:
# Free GPU memory before conversion (CPU-bound)
del merged_model, model, base_model
gc.collect()
torch.cuda.empty_cache()

!rm -rf llama.cpp
!git clone --depth 1 https://github.com/ggerganov/llama.cpp
!pip install -q -r llama.cpp/requirements.txt

In [ ]:
!python llama.cpp/convert_hf_to_gguf.py {MERGED_DIR} --outfile {GGUF_F16} --outtype f16
!ls -lh {GGUF_F16}

## Step 7: Quantize GGUF → Q4_K_M

In [ ]:
!cmake -S llama.cpp -B llama.cpp/build -DGGML_CUDA=OFF
!cmake --build llama.cpp/build --config Release -j --target llama-quantize

In [ ]:
!./llama.cpp/build/bin/llama-quantize {GGUF_F16} {GGUF_Q4} Q4_K_M
!ls -lh {GGUF_F16} {GGUF_Q4}

## Done

| Artifact | Path |
|---|---|
| LoRA adapter | `lora-adapter/` |
| Merged BF16 HF model | `model-merged/` |
| GGUF F16 | `model-f16.gguf` |
| GGUF Q4_K_M | `model-q4_k_m.gguf` |